# Previsão de risco de defasagem - Passos Mágicos

Notebook da **Pergunta 9** do Datathon. O objetivo é estimar a probabilidade de um aluno estar defasado no ano seguinte, usando somente informações disponíveis no ano corrente.

**Protocolo temporal:** treino/seleção em 2022→2023 e teste final em 2023→2024. Isso evita o vazamento que ocorreria ao prever a defasagem do mesmo ano usando IAN ou a própria defasagem.

## 1. Configuração

A base pode estar em `data/` ou ser informada pela variável de ambiente `DATATHON_DATA_PATH`. O código reutilizável está em `src/train_model.py`.

In [1]:
from pathlib import Path
import json, os, sys
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train_model import (MODEL_FEATURES, classification_metrics, dataset_summary, prepare_datasets, save_outputs, train_compare)
from predict import load_model, predict_risk

default_data = PROJECT_ROOT / 'data' / 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
DATA_PATH = Path(os.environ.get('DATATHON_DATA_PATH', default_data))
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), f'Base não encontrada: {DATA_PATH}'
DATA_PATH

WindowsPath('C:/Users/bruno/Downloads/Datathon/DATATHON/BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

## 2. Construção da base longitudinal

Cada registro de treino representa um par aluno-ano. As features vêm do ano corrente e o alvo vem exclusivamente do ano seguinte:

`target = 1 se defasagem(t+1) < 0; caso contrário, 0`.

In [2]:
train, temporal_test, all_labeled = prepare_datasets(DATA_PATH)
summary = dataset_summary(train, temporal_test)
pd.DataFrame(summary).T

,n,positives,positive_rate
train_2022_to_2023,600.0,366.0,0.610000
test_2023_to_2024,765.0,308.0,0.402614
overlap_ra_train_test,468.0,468.0,468.000000


## 3. Feature engineering

Foram criadas variáveis conhecidas no momento da previsão: tempo de programa, risco corrente, diferença entre autoavaliação e desempenho, média IDA/IEG, menor indicador e quantidade de indicadores ausentes.

`INDE` e `Pedra` foram excluídos por serem derivados dos próprios indicadores. `IPP` foi excluído porque está completamente ausente em 2022. Nome, RA e identificadores de avaliadores não são utilizados como features.

In [3]:
print(f'{len(MODEL_FEATURES)} features:')
display(pd.Series(MODEL_FEATURES, name='feature').to_frame())
display(train[MODEL_FEATURES].isna().mean().sort_values(ascending=False).to_frame('taxa_ausencia'))

17 features:


,feature
0,idade
1,anos_programa
2,iaa
3,ieg
4,ips
5,ida
6,ipv
7,ian
8,defasagem_atual
9,risco_atual


,taxa_ausencia
idade,0.0
anos_programa,0.0
iaa,0.0
ieg,0.0
ips,0.0
ida,0.0
ipv,0.0
ian,0.0
defasagem_atual,0.0
risco_atual,0.0


## 4. Divisão de treino e teste

A divisão é cronológica e não aleatória. O teste 2023→2024 permanece intocado durante a seleção da arquitetura e dos hiperparâmetros.

In [4]:
class_balance = pd.DataFrame({
    'treino_2022_2023': train['target_risco_proximo_ano'].value_counts(normalize=True),
    'teste_2023_2024': temporal_test['target_risco_proximo_ano'].value_counts(normalize=True),
}).fillna(0).sort_index()
class_balance.index = ['sem_risco', 'com_risco']
class_balance

,treino_2022_2023,teste_2023_2024
sem_risco,0.39,0.597386
com_risco,0.61,0.402614


## 5. Treinamento e seleção

Comparamos Regressão Logística, Random Forest e XGBoost. Cada pipeline inclui imputação, padronização numérica e one-hot encoding. A arquitetura é avaliada por validação cruzada estratificada de 5 folds no treino e pela estabilidade fora do tempo. A regra conservadora escolhe o maior valor de `min(ROC-AUC CV, ROC-AUC temporal)`, evitando publicar um modelo que colapse com a mudança de período. Por participar da seleção, 2023→2024 é chamado de validação temporal, não de teste final intocado.

O limiar de decisão é estimado com previsões out-of-fold, maximizando F2 para priorizar recall.

In [5]:
selected_name, evaluated_pipeline, threshold, comparison = train_compare(train, temporal_test)
print('Modelo selecionado:', selected_name)
print('Limiar selecionado no treino:', round(threshold, 4))

Modelo selecionado: xgboost
Limiar selecionado no treino: 0.1351


In [6]:
rows = []
for model_name, values in comparison.items():
    rows.append({
        'modelo': model_name,
        'cv_precision': values['cv_precision'],
        'cv_recall': values['cv_recall'],
        'cv_roc_auc': values['cv_roc_auc'],
        **{f'test_{k}': v for k, v in values['temporal_test'].items()            if k in ['precision','recall','f1','roc_auc','pr_auc','brier_score']},
    })
results = pd.DataFrame(rows).set_index('modelo').sort_values('cv_roc_auc', ascending=False)
results.round(3)

,cv_precision,cv_recall,cv_roc_auc,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_brier_score
modelo,,,,,,,,,
regressao_logistica,0.858,0.789,0.874,0.412,0.386,0.399,0.452,0.365,0.40
xgboost,0.853,0.776,0.862,0.529,0.935,0.676,0.836,0.800,0.16
random_forest,0.835,0.776,0.852,0.541,0.961,0.692,0.822,0.773,0.17


In [7]:
plot_data = results[['test_precision','test_recall','test_roc_auc']].reset_index()
plot_data = plot_data.melt(id_vars='modelo', var_name='metrica', value_name='valor')
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x='modelo', y='valor', hue='metrica')
plt.ylim(0, 1)
plt.title('Desempenho no teste temporal 2023→2024')
plt.ylabel('Valor')
plt.xlabel('')
plt.tight_layout()
plt.show()

C:\Users\bruno\AppData\Local\Temp\ipykernel_20096\4011198696.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Validação temporal do modelo selecionado

Precision mede a proporção de alertas corretos; recall mede quantos alunos realmente em risco foram identificados; ROC-AUC avalia a ordenação dos riscos em todos os limiares. PR-AUC e Brier Score complementam a análise.

In [8]:
selected_metrics = comparison[selected_name]['temporal_test']
display(pd.Series(selected_metrics).drop('confusion_matrix').to_frame('valor'))
cm = pd.DataFrame(selected_metrics['confusion_matrix'],
                  index=['real_sem_risco','real_com_risco'],
                  columns=['prev_sem_risco','prev_com_risco'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de confusão - {selected_name}')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()

,valor
threshold,0.135148
precision,0.529412
recall,0.935065
f1,0.676056
roc_auc,0.835701
pr_auc,0.79951
brier_score,0.160481
positive_rate,0.402614
predicted_positive_rate,0.711111
n,765


C:\Users\bruno\AppData\Local\Temp\ipykernel_20096\4009955470.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Modelo final e serialização

Depois da avaliação, a arquitetura selecionada é reajustada com todos os pares rotulados (2022→2023 e 2023→2024). O arquivo Joblib contém pipeline, limiar e metadados para que o Streamlit execute exatamente o mesmo pré-processamento.

In [9]:
saved = save_outputs(MODEL_DIR, selected_name, evaluated_pipeline, threshold, comparison,
                     train, temporal_test, all_labeled)
(REPORT_DIR / 'model_comparison.json').write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print('Modelo salvo em:', saved['model_path'])

Modelo salvo em: C:\Users\bruno\Downloads\codex\datathon_ml\models\modelo_risco_defasagem.joblib


## 8. Teste de consumo pela aplicação Web

O teste abaixo recarrega o arquivo salvo e realiza inferência em registros no mesmo formato que será enviado pela aplicação.

In [10]:
bundle = load_model(MODEL_DIR / 'modelo_risco_defasagem.joblib')
example = temporal_test[MODEL_FEATURES].head(5).copy()
predictions = predict_risk(bundle, example)
predictions[['probabilidade_risco','risco_previsto']]

,probabilidade_risco,risco_previsto
0,0.644235,True
1,0.825495,True
2,0.801489,True
3,0.741056,True
4,0.666028,True


## 9. Limitações e uso responsável

- Há apenas duas transições anuais rotuladas; as métricas devem ser monitoradas quando 2025 estiver disponível.
- O modelo é um mecanismo de triagem, não uma decisão automática.
- Probabilidades não devem ser usadas para excluir alunos de atividades ou benefícios.
- Recomenda-se monitorar recall, precisão, calibração e resultados por grupos.
- Mudanças na coleta dos indicadores exigem validação e possível retreinamento.